# Practice Lab: Math Foundations 3 - Transformer Architecture

8 exercises on matmul, attention, and activation functions.

In [ ]:
!pip install -q numpy torch matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
print('torch:', torch.__version__)

## Exercise 1 - Matmul by hand

In [ ]:
A = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
B = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
C = A @ B
print(C)
# Expected: tensor([[4., 5.], [10., 11.]])

## Exercise 2 - Identity QK^T

In [ ]:
Q = torch.eye(4)
K = torch.eye(4)
scores = Q @ K.T / math.sqrt(4)
weights = torch.softmax(scores, dim=-1)
print(weights.round(decimals=3))

## Exercise 3 - Linear collapse

In [ ]:
import numpy as np
W1 = np.array([[1, 0], [1, 1]])
W2 = np.array([[2, 1], [0, 1]])
x  = np.array([1, 2])

# Two linear layers vs the single combined matrix W2 @ W1
print('two layers:', W2 @ (W1 @ x))     # [5 3]
print('combined:  ', (W2 @ W1) @ x)     # [5 3]  - identical

# Insert a ReLU between them, with x = [1, -3]
x2 = np.array([1, -3])
h  = np.maximum(0, W1 @ x2)              # ReLU clips the negative
print('with ReLU: ', W2 @ h)            # [2 0]
print('collapsed: ', (W2 @ W1) @ x2)    # [0 -2]  - no single matrix matches

## Exercise 4 - Causal mask correctness

In [ ]:
torch.manual_seed(0)
N, d = 5, 4
Q = torch.randn(N, d)
K = torch.randn(N, d)
V = torch.randn(N, d)

scores = Q @ K.T / math.sqrt(d)
mask = torch.tril(torch.ones(N, N))
scores = scores.masked_fill(mask == 0, float('-inf'))
weights = torch.softmax(scores, dim=-1)
print(weights.round(decimals=3))
assert (weights.triu(diagonal=1) == 0).all()

## Exercise 5 - sqrt(d_k) variance check

In [ ]:
for d_k in [4, 16, 64, 256]:
    Q = torch.randn(1000, d_k)
    K = torch.randn(1000, d_k)
    raw = (Q @ K.T)
    scaled = raw / math.sqrt(d_k)
    print(f'd_k={d_k:4d}  raw_var={raw.var().item():.2f}  scaled_var={scaled.var().item():.2f}')

## Exercise 6 - Activation comparison

In [ ]:
x = torch.linspace(-3, 3, 200)
relu = F.relu(x); gelu = F.gelu(x); silu = F.silu(x)
plt.plot(x.numpy(), relu.numpy(), label='ReLU', color='red')
plt.plot(x.numpy(), gelu.numpy(), label='GELU', color='blue')
plt.plot(x.numpy(), silu.numpy(), label='SiLU (swish)', color='purple')
plt.legend(); plt.title('Activations'); plt.show()
for xv in [-1.0, 0.0, 1.0]:
    t = torch.tensor([xv])
    print(f'x={xv:+.1f}  ReLU={F.relu(t).item():.3f}  GELU={F.gelu(t).item():.3f}  SiLU={F.silu(t).item():.3f}')

## Exercise 7 - Implement scaled dot-product attention

In [ ]:
def my_sdpa(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = torch.softmax(scores, dim=-1)
    return weights @ V

torch.manual_seed(0)
Q = torch.randn(4, 8)
K = torch.randn(4, 8)
V = torch.randn(4, 8)
mine = my_sdpa(Q, K, V)
ref = F.scaled_dot_product_attention(Q, K, V)
print(f'max abs diff: {(mine - ref).abs().max().item():.2e}')
assert torch.allclose(mine, ref, atol=1e-5)

## Exercise 8 - Multi-head extension

In [ ]:
def my_multihead(x, qkv_weight, out_weight, n_heads):
    B, T, D = x.shape
    d_head = D // n_heads
    qkv = x @ qkv_weight.T
    q, k, v = qkv.chunk(3, dim=-1)
    q = q.view(B, T, n_heads, d_head).transpose(1, 2)
    k = k.view(B, T, n_heads, d_head).transpose(1, 2)
    v = v.view(B, T, n_heads, d_head).transpose(1, 2)
    scores = q @ k.transpose(-2, -1) / math.sqrt(d_head)
    weights = torch.softmax(scores, dim=-1)
    attn = weights @ v
    out = attn.transpose(1, 2).contiguous().view(B, T, D)
    return out @ out_weight.T

torch.manual_seed(0)
B, T, D, H = 2, 6, 32, 4
x = torch.randn(B, T, D)
qkv_w = torch.randn(3*D, D)
out_w = torch.randn(D, D)
result = my_multihead(x, qkv_w, out_w, H)
print(f'output shape: {result.shape}')
assert result.shape == (B, T, D)

## Wrap-up

Exercise 4 (causal mask) and Exercise 7 (single-head from scratch) make the math click. Lesson 1.4 adds the plumbing.